# Proof of concept: camera

Can a policy learn to drive one track from camera pixels alone? One track, no
domain randomization, no holdout. An HPO over PPO knobs, network shape,
resolution, frame stack and the reward design, then one long run whose metrics
are the proof.

The feature-vector half is `proof_feature.ipynb`, and it is already solved:
100% lap completion, 0% off-track. Run it first if you have not; this notebook
inherits its finding about which rewards are worth searching.

In [ ]:
import inspect
import json
import os

# Before any import touches CUDA. Madrona's default device heap kills the
# kernel outright on a 16 GB card -- no exception, no Optuna FAIL, the process
# is simply gone. Expandable segments stop a failed trial from fragmenting the
# allocator for the next one.
os.environ.setdefault("MADRONA_MWGPU_DEVICE_HEAP_SIZE", str(1 << 30))
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
from deepracer_genesis.experiment import (
    AsymmetricCameraPolicy,
    CameraEnvironment,
    Evaluation,
    PPO,
    RewardShaping,
    run,
)
from deepracer_genesis.randomization.spaces import Choice, FloatRange, IntRange
import optuna
import torch

from experiments import rewards

In [ ]:
SEED = 0
ROOT = "runs/proof_camera"
TRACK = "reinvent_base"

# Episodes end on the task, not the clock: 3 laps truncates (bootstrapped,
# never penalized). max_laps counts finish-line CROSSINGS, and random_start
# spawns mid-lap, so the distance covered averages 2.5 laps rather than 3.
MAX_LAPS = 3
EPISODE_LENGTH_S = 60.0

# 32 envs costs ~3x aggregate throughput against 128 (measured 5,995 vs 18,262
# agg steps/s, benchmarks/results.csv) but buys back the whole resolution axis,
# because rollout storage scales with envs.
CAMERA_ENVS = 32
RESOLUTION = (160, 120)            # physical camera parity; the 1x base
FRAME_STACK = 1                    # default only; the study samples it

# Both camera axes are searched. Estimated peak VRAM at 32 envs (2x rollout obs
# plus ~2 GB of madrona heap, genesis buffers and context):
#           160x120  320x240  480x360
#   fs=2      2.7      4.8      8.4
#   fs=3      3.1      6.2     11.6
#   fs=4      3.4      7.7     14.7  <- over budget
#   fs=5      3.8      9.1     17.9  <- over budget
# The preflight prunes the two over-budget draws before any scene is built.
# 160x120 is the only resolution the physical car runs, so a winner above 1x is
# a research result rather than a deployable one.
RES_SCALES = (1, 3)
FRAME_STACKS = (2, 5)
VRAM_BUDGET_GB = 13.0

# The six rewards that ever reached a lap in the 200-trial feature study. The
# twelve dropped ones never completed once; most are the 1e5-scale families
# whose value loss explodes, which is modality-independent. Set this to
# list(REWARDS) to search all eighteen again.
REWARD_NAMES = ["classic_center_line", "center_line_with_direction_guard",
                "lap_bonus_baseline", "mitigate_zig_zag_with_granular",
                "single_aws_reward", "exponential_center_second_model"]

# ~18 draws per reward arm. At an estimated 1,379 steps/s (32 envs at 1x) this
# is roughly 33 h nominal and nearer 23 h once Hyperband prunes; the 2x and 3x
# draws are slower still. Three eval rungs give the pruner room to work.
CAMERA_TRIALS = 108
CAMERA_TRIAL_STEPS = 1_500_000
CAMERA_TRIAL_EVAL_EVERY = 500_000

# A draw under this at any eval has collapsed, so kill it. Validated on the
# feature study: all 8 later-eval kills were monotonically declining.
CAMERA_TAKEOFF_MIN_M = 2.0

CAMERA_FINAL_STEPS = 15_000_000
CAMERA_FINAL_EVAL_EVERY = 1_000_000

os.makedirs(ROOT, exist_ok=True)

In [ ]:
# live curves for every trial and the final run; each run dir is its own TB run
%load_ext tensorboard
%tensorboard --logdir $ROOT

## Rewards

The collection is `experiments/rewards.py`, a torch port of
[DeepRacerRewardFunctionsCollection](https://github.com/Machine-Learning-Labs/DeepRacerRewardFunctionsCollection).
`lap_bonus_baseline` is the control: it pays 1.0 only on a finish-line
crossing. `run()` seeds python, numpy and torch from `spec.seed`, so `SEED`
alone determines the run; set `DR_DETERMINISTIC=1` for deterministic kernels.

In [ ]:
#: every reward defined in experiments/rewards.py, in definition order
REWARDS = {name: fn for name, fn in vars(rewards).items()
           if inspect.isfunction(fn) and fn.__module__ == rewards.__name__}

print(f"{len(REWARDS)} rewards available")
print(f"searching {len(REWARD_NAMES)}: {REWARD_NAMES}")

In [ ]:
SEARCH_SPACE = {
    "lr": FloatRange(1e-4, 1e-2, log=True),
    "entropy_coef": FloatRange(1e-3, 3e-2, log=True),
    "epochs": IntRange(3, 8),
    "clip": FloatRange(0.1, 0.3),
    "gamma": Choice([0.98, 0.99, 0.995, 0.999]),
    "minibatches": Choice([2, 4, 8]),
}

# fixed taper for the layers after the first; at depth 3, k0=8, s0=4 this is the
# stock (8, 4, 3) / (4, 2, 1) trunk
CNN_TAPER_K = (4, 3, 3)
CNN_TAPER_S = (2, 1, 1)


def suggest_reward(trial: optuna.Trial) -> str:
    """Samples which reward to train on.

    Args:
        trial: The Optuna trial.

    Returns:
        A REWARDS key.
    """
    return trial.suggest_categorical("reward", REWARD_NAMES)


def suggest_frame_stack(trial: optuna.Trial) -> int:
    """Samples how many camera frames are stacked along the channel axis.

    Args:
        trial: The Optuna trial.

    Returns:
        The ``frame_stack`` for ``CameraEnvironment``.
    """
    return trial.suggest_int("frame_stack", *FRAME_STACKS)


def suggest_resolution(trial: optuna.Trial) -> tuple:
    """Samples the camera resolution as an integer multiple of the 1x base.

    The multiplier is drawn rather than the (w, h) pair because Optuna's sqlite
    storage only persists scalar categoricals.

    Args:
        trial: The Optuna trial.

    Returns:
        The ``(width, height)`` tuple for ``CameraEnvironment``.
    """
    scale = trial.suggest_int("res_scale", *RES_SCALES)
    return (RESOLUTION[0] * scale, RESOLUTION[1] * scale)


def vram_estimate_gb(resolution: tuple, frame_stack: int, num_envs: int) -> float:
    """Predicts a camera trial's peak VRAM in GB, before the scene is built.

    Args:
        resolution: Camera ``(width, height)``.
        frame_stack: Frames stacked along the channel axis.
        num_envs: Parallel environments.

    Returns:
        Twice the rollout observation, plus fixed non-torch overhead.
    """
    width, height = resolution
    rollout = 24 * num_envs * 3 * frame_stack * height * width * 4 / 1e9
    return 2.0 * rollout + 2.0


def suggest_mlp(trial: optuna.Trial) -> tuple:
    """Samples the MLP head as a halving funnel of layer widths.

    Args:
        trial: The Optuna trial.

    Returns:
        The ``hidden`` tuple for the policy's ``mlp`` dict.
    """
    depth = trial.suggest_int("depth", 2, 4)
    width = trial.suggest_categorical("width", [128, 256, 512])
    return tuple(max(width // 2 ** i, 32) for i in range(depth))


def suggest_cnn(trial: optuna.Trial) -> dict:
    """Samples the CNN trunk, deriving all three tuples from one depth draw.

    Drawing depth once makes a channels/kernels/strides length mismatch
    unrepresentable rather than an IndexError several trials in.

    Args:
        trial: The Optuna trial.

    Returns:
        The ``cnn`` dict for ``AsymmetricCameraPolicy``.
    """
    depth = trial.suggest_int("cnn_depth", 2, 4)
    k0 = trial.suggest_categorical("k0", [8, 5, 3])
    s0 = trial.suggest_categorical("s0", [4, 2])
    return {
        "channels": tuple(16 * 2 ** min(i, 2) for i in range(depth)),
        "kernels": (k0,) + CNN_TAPER_K[:depth - 1],
        "strides": (s0,) + CNN_TAPER_S[:depth - 1],
        "activation": "relu",
    }

In [ ]:
def camera_spec(*, lr, entropy_coef, epochs, clip, gamma, minibatches, mlp, cnn,
                reward, steps, eval_every, variant, resolution=RESOLUTION,
                frame_stack=FRAME_STACK, group="proof_camera"):
    """Builds one camera proof spec.

    The single place a spec is built, so the winning trial and the final run
    cannot drift apart.

    Args:
        lr: PPO learning rate.
        entropy_coef: PPO entropy bonus coefficient.
        epochs: PPO optimization epochs per iteration.
        clip: PPO surrogate clipping range.
        gamma: Reward discount factor.
        minibatches: Minibatches per PPO epoch.
        mlp: ``hidden`` tuple for the policy head.
        cnn: The ``cnn`` dict for the policy trunk.
        reward: A REWARDS key naming which reward to train on.
        steps: Total env-steps, a spec field rather than a stage argument.
        eval_every: Periodic deterministic-eval cadence in env-steps.
        variant: Run-dir variant tag.
        resolution: Camera ``(width, height)`` in pixels.
        frame_stack: Camera frames stacked along the channel axis.
        group: Run-dir group tag.

    Returns:
        The validated frozen ExperimentSpec.
    """
    return (
        CameraEnvironment(
            num_envs=CAMERA_ENVS,
            tracks=(TRACK,),
            resolution=resolution,
            frame_stack=frame_stack,
            max_laps=MAX_LAPS,
            episode_length_s=EPISODE_LENGTH_S,
        )
        >> AsymmetricCameraPolicy(
            actor_keys=("camera",),
            critic_keys=("camera", "state"),
            cnn=cnn,
            mlp={"hidden": mlp, "activation": "relu"},
            distribution={"std_range": (0.1, 1.0)},
        )
        # every reward returns the step reward itself, so scales stay empty
        >> RewardShaping(fn=REWARDS[reward], scales={})
        >> PPO(lr=lr, entropy_coef=entropy_coef, epochs=epochs, clip=clip,
               gamma=gamma, minibatches=minibatches)
        >> Evaluation(charts=True)
    ).build(seed=SEED, total_env_steps=steps, eval_every_steps=eval_every,
            group=group, variant=variant)

## HPO

Trials are ranked on `completion_rate`, ground truth the reward shaping cannot
inflate, with `mean_progress_m` as the tie-break. The tie-break carries the
ranking early: camera completion sits at 0 for most of a trial budget.

Three stops per trial. A VRAM preflight rejects a draw before the scene is
built, an absolute floor kills a collapsed draw, and Hyperband ranks the rest.

In [ ]:
def make_study(name: str, trial_steps: int, eval_every: int) -> optuna.Study:
    """Creates or resumes one seeded, pruning, sqlite-backed study.

    Args:
        name: Study name; also its row in ``<ROOT>/study.db``.
        trial_steps: Max env-steps per trial, the pruner's max resource.
        eval_every: Eval cadence, the pruner's min resource.

    Returns:
        The Optuna study.
    """
    return optuna.create_study(
        direction="maximize",
        study_name=name,
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.HyperbandPruner(
            min_resource=eval_every, max_resource=trial_steps,
            reduction_factor=3),
        storage=f"sqlite:///{ROOT}/study.db",
        load_if_exists=True,
    )


def stop(trial: optuna.Trial, outcome: str, reason: str):
    """Records why a trial stopped, then prunes it.

    Optuna persists neither a ``TrialPruned`` message nor a caught exception's
    text, so the reason survives only as a user attribute.

    Args:
        trial: The Optuna trial.
        outcome: Machine-readable tag, e.g. "parked" or "oom_predicted".
        reason: Human-readable detail.

    Raises:
        optuna.TrialPruned: Always.
    """
    trial.set_user_attr("outcome", outcome)
    trial.set_user_attr("reason", reason)
    raise optuna.TrialPruned(f"{outcome}: {reason}")


def report_to(trial: optuna.Trial, takeoff_min: float):
    """Builds the periodic-eval callback that stops a trial early.

    Two independent stops. ``mean_progress_m`` is what Hyperband ranks on
    because ``completion_rate`` sits at 0 for most of a trial budget and cannot
    discriminate. The absolute floor kills a draw that has collapsed, which
    Hyperband cannot do without peers to rank it against.

    Args:
        trial: The Optuna trial.
        takeoff_min: Mean progress (m) a live draw must hold at every eval.

    Returns:
        An ``on_eval(frames, metrics)`` callback.

    Raises:
        optuna.TrialPruned: Raised inside the callback, which stops the run.
    """
    def report(frames: int, metrics: dict) -> None:
        progress = metrics["mean_progress_m"]
        trial.report(progress, frames)
        if progress < takeoff_min:
            stop(trial, "parked",
                 f"{progress:.2f} m at {frames:,} steps, floor {takeoff_min} m")
        if trial.should_prune():
            stop(trial, "hyperband",
                 f"ranked below peers at {frames:,} steps ({progress:.2f} m)")

    return report


def score(metrics: dict) -> float:
    """Ranks a finished trial: completion first, progress as the tie-break.

    Args:
        metrics: The final deterministic eval's metrics.

    Returns:
        The objective value to maximize.
    """
    return float(metrics["completion_rate"] * 100.0
                 + 0.05 * metrics["mean_progress_m"])


def leaderboard(study: optuna.Study) -> pd.DataFrame:
    """Tabulates a finished study, best score first.

    Args:
        study: The Optuna study.

    Returns:
        One row per trial.
    """
    return pd.DataFrame([
        {"trial": t.number, "state": t.state.name, "score": t.value,
         "completion": t.user_attrs.get("completion"),
         "progress_m": t.user_attrs.get("progress_m"), **t.params}
        for t in study.trials]).sort_values("score", ascending=False)


def outcomes(study: optuna.Study) -> pd.DataFrame:
    """Tabulates every trial with the recorded reason it stopped.

    Args:
        study: The Optuna study.

    Returns:
        One row per trial: number, state, outcome, reason, score and params.
    """
    return pd.DataFrame([
        {"trial": t.number, "state": t.state.name,
         "outcome": t.user_attrs.get("outcome", "unrecorded"),
         "reason": t.user_attrs.get("reason", ""),
         "score": t.value, **t.params}
        for t in study.trials])


def failure_report(study: optuna.Study, name: str) -> pd.DataFrame:
    """Prints why trials stopped, overall and per reward, and returns the detail.

    A reward whose every trial parked is a finding about that reward; one whose
    trials Hyperband pruned merely lost a ranking.

    Args:
        study: The Optuna study.
        name: Label used in the printed headings.

    Returns:
        The rows for trials that did not complete.
    """
    table = outcomes(study)
    print(f"=== {name}: {len(table)} trials ===")
    print(table["outcome"].value_counts().to_string(), "\n")
    if "reward" in table:
        print((table.groupby("reward")["outcome"]
               .value_counts().unstack(fill_value=0)).to_string(), "\n")
    return table[table["outcome"] != "completed"]


optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Trains one sampled camera config and scores it on completion.

    Args:
        trial: The Optuna trial.

    Returns:
        The trial score: completion, with progress as the tie-break.

    Raises:
        optuna.TrialPruned: On an over-budget draw, an OOM, or a Hyperband cut.
    """
    cfg = {name: space.suggest(trial, name)
           for name, space in SEARCH_SPACE.items()}
    cfg["reward"] = suggest_reward(trial)
    cfg["resolution"] = suggest_resolution(trial)
    cfg["frame_stack"] = suggest_frame_stack(trial)
    cfg["mlp"] = suggest_mlp(trial)
    cfg["cnn"] = suggest_cnn(trial)
    trial.set_user_attr("cfg", cfg)

    need = vram_estimate_gb(cfg["resolution"], cfg["frame_stack"], CAMERA_ENVS)
    if need > VRAM_BUDGET_GB:
        stop(trial, "oom_predicted",
             f"{need:.1f} GB needed > {VRAM_BUDGET_GB} GB budget")

    try:
        record = run(camera_spec(**cfg, steps=CAMERA_TRIAL_STEPS,
                                 eval_every=CAMERA_TRIAL_EVAL_EVERY,
                                 group="proof_camera_hpo",
                                 variant=f"t{trial.number:03d}"),
                     root=ROOT, on_eval=report_to(trial, CAMERA_TAKEOFF_MIN_M))
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        stop(trial, "oom_runtime", str(exc).split(".")[0][:120])

    trial.set_user_attr("outcome", "completed")
    trial.set_user_attr("reason", "")
    trial.set_user_attr("completion", record.metrics["completion_rate"])
    trial.set_user_attr("progress_m", record.metrics["mean_progress_m"])
    return score(record.metrics)


study = make_study("proof_camera_v2", CAMERA_TRIAL_STEPS,
                   CAMERA_TRIAL_EVAL_EVERY)
study.optimize(objective, n_trials=CAMERA_TRIALS, catch=(RuntimeError,))

display(leaderboard(study))
BEST = study.best_trial.user_attrs["cfg"]
print("winner:", BEST)

## What did not work

Optuna keeps neither a `TrialPruned` message nor a caught exception's text, so
each trial records an `outcome` tag and a `reason` before it stops. The tags are
`completed`, `parked` (collapsed below the floor), `hyperband` (ranked below
peers), `oom_predicted` (rejected by the preflight), `oom_runtime`, and
`unrecorded` (a crash caught by `catch=(RuntimeError,)`).

In [ ]:
FAILED = failure_report(study, "camera")
display(FAILED[["trial", "outcome", "reason", "reward"]])

## Final training

The winning config, trained long. Camera takeoff is bistable, so if this returns
0% completion with short episodes, re-run the cell for a fresh draw.

In [ ]:
FINAL = run(camera_spec(**BEST, steps=CAMERA_FINAL_STEPS,
                        eval_every=CAMERA_FINAL_EVAL_EVERY, variant="final"),
            root=ROOT)
print({k: round(v, 3) for k, v in FINAL.metrics.items() if isinstance(v, float)})

## Result

`completion_rate` is the claim. `eval_history` shows it was learned rather than
lucky. The table picks up `proof_feature.ipynb`'s row if that notebook has run.

In [ ]:
def proof_row(record, cfg: dict) -> dict:
    """Builds this notebook's row of the proof table.

    Args:
        record: The finished run's EvalRecord.
        cfg: The winning config that produced the run.

    Returns:
        A flat dict of the metrics the proof claims.
    """
    return {
        "modality": "camera",
        "reward": cfg["reward"],
        "completion_rate": record.metrics["completion_rate"],
        "mean_laps": record.metrics["mean_laps"],
        "lap_time_s": record.metrics["lap_time_s"],
        "mean_progress_m": record.metrics["mean_progress_m"],
        "offtrack_rate": record.metrics["offtrack_rate"],
        "mean_speed_mps": record.metrics["mean_speed_mps"],
        "env_steps": record.train["total_env_steps"],
        "wall_clock_s": record.train["wall_clock_s"],
        "steps_per_s": record.train["steps_per_s"],
    }


ROW = proof_row(FINAL, BEST)
with open(os.path.join(ROOT, "proof_row.json"), "w") as f:
    json.dump(ROW, f, indent=2)

# the other notebook's row, if it has already run
rows = [ROW]
for other in ("runs/proof/proof_row.json", "runs/proof_camera/proof_row.json"):
    if os.path.exists(other) and other != os.path.join(ROOT, "proof_row.json"):
        rows.append(json.load(open(other)))
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(7, 4))
history = pd.DataFrame(FINAL.eval_history)
ax.plot(history["frames"], history["completion_rate"], marker="o",
        label="camera")
ax.set_xlabel("env steps")
ax.set_ylabel("completion rate")
ax.set_title(f"Learning to lap {TRACK}")
ax.legend()
fig.tight_layout()